# Contrastive set — activation extraction (Colab GPU)

Extracts residual-stream activations for the `demo16_v2` contrastive items **and the frozen lexical controls**, so a linear eval/deploy probe can be fitted and interrogated offline.

**What this run is for.** The first run (`RESULT_8b.md`) settled the question that was blocking the cue-locus decision: the held-out reader gate's difficulty ordering does not appear in linear decodability at any depth, on either model, in either language. It raised two new questions that this run is built to settle.

1. **Is the direction reading purpose, or the scrutiny-versus-use verbs?** Every cue locus that decoded well carried a verb contrast; the only locus isolating the agent noun failed at chance. That is what a verb-driven direction looks like. The `implicit_cue` lexical controls separate the two and had never been extracted — so they are extracted here.
2. **Which layer and token position get frozen?** Layer 8 of 32 won every metric and was the only depth to pass the per-fold gate, but it was the *shallowest of four points sampled*. This run samples the shallow end densely (layers 2–16, plus 24 and 32) and stores three read-out positions rather than one.

**Why GPU matters here.** On CPU only a ~1B stand-in is feasible, which supports a qualitative claim at best. On a Colab GPU you can run **8B models in the actual subject lineage**, including a Japanese-specialised one — that makes the Japanese arm meaningful rather than a floor.

| GPU | what fits |
|---|---|
| T4 16GB | 8B in 4-bit |
| L4 24GB | 8B in fp16 |
| A100 40GB | 8B in fp16 comfortably; 70B in 4-bit is borderline — try `LOAD_IN_4BIT` and a batch size of 1–2 |

**Nothing here is gate evidence.** `demo16_v2` is a development corpus the frozen protocol forbids reusing, and these are not the declared subjects (Llama-3.3-70B, Swallow-70B, Stockmark-2-100B). This is a diagnostic only.

**`naturalistic_transfer.jsonl` is deliberately not loaded.** The protocol allows that set to be scored once, after the probe and every analytical choice are final. Nothing in the current question needs it, and pulling it in only creates opportunities to spend it early.

---
### How to run

Everything moves through **Drive** at `MyDrive/lang-ea` — cell 3 searches it for inputs, cell 5 writes the results there. That survives runtime restarts, which per-session uploads do not.

**Inputs.** Put these seven files in `MyDrive/lang-ea`, keeping the filenames exactly as they are (cell 3 looks them up by basename, in `.`, then `/content`, then Drive):

- `contrastive/data/processed/demo16_v2/probe_items.jsonl`
- `contrastive/data/processed/demo16_v2/probe_items.newframes.jsonl`
- `contrastive/data/processed/demo16_v2/probe_items.v1_judge.jsonl`
- `contrastive/data/processed/demo16_v2/probe_items.v2_examiner.jsonl`
- `contrastive/data/processed/demo16_v2/probe_items.v3_critic.jsonl`
- `contrastive/purpose_frames.yaml`
- `contrastive/data/processed/demo16_v2/lexical_controls.jsonl`

Every set after the first is the same 1200 payloads with one line substituted, so payload, folds, container, generator and every hash-bearing field stay byte-identical. Cell 3 asserts they differ *only* in that first line and stops if they do not, so a broken comparison is caught before the GPU time is spent.

- `.newframes` — catalog 2.3.2. Japanese `noun_only` improved in 15/16 cells; English did not (4/16) and the EN–JA gap widened from 0.14 to 0.21. This is the **baseline the variants are measured against**, not the base corpus.
- `.v1_judge` / `.v2_examiner` / `.v3_critic` — candidate English rewrites of the p06 and p09 **eval** members, built from the 2.3.2 set by `probe_check/development/baseline/build_variant_items.py`. Japanese and both deploy members are held fixed, so each variant is a one-factor test of the English agent noun. The pairs are scored separately, so this single pass yields three candidates for p06 *and* three for p09.

**What decides the winner.** Read the `noun_only` row per pair at `--position last`, layers 2–16, on both models, and prefer the effect-size column — the fitted direction shifts slightly between sets because the training set changed, which moves even byte-identical items by up to 0.06 balanced accuracy (median |Δ| 0.032, p90 0.071). A candidate has to beat that noise floor on both models to count. The target is English `noun_only` ≥ 0.65 with the EN–JA gap *narrower* than 2.3.2's 0.21, not merely English improving.

Drop entries from `ITEM_SETS` in cell 2 to save time or space; the controls attach to whichever set is listed first.

Do that once and no per-session upload is needed. For a one-off instead: from VS Code, connect a GPU runtime via **Select Kernel**, then right-click each file in the Explorer → **"Upload to Colab Session"** (Command Palette: *Colab: Upload File to Runtime*); from browser Colab, drag them into the file tree root via the **folder icon** in the sidebar.

`google.colab.files.upload()` and `files.download()` both fail under the VS Code extension, which cannot reach the local filesystem (googlecolab/colab-vscode#276). Cells 3 and 5 catch that and fall back, so neither is a blocker.

**Run.** Set a GPU runtime, then run every cell top to bottom. Cell 3 prints the resolved path and size of each input, the item counts, and checks the control counts against what the contract expects — so a truncated file is caught before the GPU time is spent rather than after.

**Output.** Cell 5 writes `activations.zip` to `MyDrive/lang-ea/`, overwriting any previous run's archive. Each model produces one directory per item set — `out/<model>` and `out/<model>_newframes` — and the lexical controls attach to the first as `out/<model>/controls`. `analyze.py --acts <dir>` reads any of them unchanged.

At the default settings that is ~590 MB per model, so ~1.2 GB in total, and float16 tensors barely compress — the zip is close to the raw size. Drop `"mean"` from `POSITIONS` first if that is a problem: it is the least likely position to win, since averaging over a long payload dilutes a frame that occupies one line, and removing it takes the archive to ~800 MB. Copy the archive to `contrastive/probe_check/out/` locally and unzip.

In [ ]:
#@title 1. Check GPU and install
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U "transformers>=4.44" accelerate bitsandbytes pyyaml 2>&1 | tail -2
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Set Runtime -> Change runtime type -> GPU"

In [ ]:
#@title 2. Configuration

# Models are chosen to sit in the SAME LINEAGE as the declared subject models
# (Llama-3.3-70B, Llama-3.3-Swallow-70B, Stockmark-2-100B). Swallow-8B is the
# small sibling of subject #2 and is Japanese-specialised, which is what makes
# the Japanese arm interpretable rather than a floor.
MODELS = [
    "tokyotech-llm/Llama-3.1-Swallow-8B-Instruct-v0.3",  # JA-specialised, subject lineage
    "unsloth/Meta-Llama-3.1-8B-Instruct",                # EN baseline, ungated mirror
]

# Item sets. Each becomes its own output directory, so `analyze.py --acts <dir>`
# reads them unchanged. The model is loaded once and reused across sets, so each
# extra set costs only its own forward passes.
#
#   ""            the corpus as generated (pre-2.3.x frames). Lexical controls
#                 attach to whichever set is listed FIRST.
#   "_newframes"  catalog 2.3.2. The measured baseline every variant below is
#                 compared against: Japanese noun_only improved in 15/16 cells,
#                 English did not improve (4/16) and the EN-JA gap widened.
#   "_v1..v3"     candidate English rewrites of the p06 and p09 eval members,
#                 built by probe_check/development/baseline/build_variant_items.py from the 2.3.2 set.
#                 Japanese and the deploy members are held fixed, so each variant
#                 is a one-factor test of the English agent noun. Each pair is
#                 scored separately, so one pass yields three candidates for p06
#                 AND three for p09.
#
# Drop entries to save time/space -- but keep at least one set, and note that the
# controls follow the first entry.
ITEM_SETS = [
    ("", "probe_items.jsonl"),
    ("_newframes", "probe_items.newframes.jsonl"),
    ("_v1_judge", "probe_items.v1_judge.jsonl"),
    ("_v2_examiner", "probe_items.v2_examiner.jsonl"),
    ("_v3_critic", "probe_items.v3_critic.jsonl"),
]

# Absolute layer indices, not fractions. The first run sampled 8/16/24/32; layer
# 8 -- the shallowest of the four -- won every metric and was the only depth to
# pass the per-fold gate. The optimum may be shallower still, or between the
# points sampled, so the shallow end is dense here. Layer choice has to be
# frozen before results are seen, and four points is not enough to freeze on.
LAYERS = [2, 4, 6, 8, 10, 12, 14, 16, 24, 32]

# The contract also requires the token position to be frozen. The first run
# stored exactly one candidate, which is not a choice.
#   last   final token of the generation prompt (what the first run used)
#   mean   mean over non-padding tokens
#
# "frame" (final token of the purpose-frame sentence) was stored by the previous
# run and is now DELIBERATELY DROPPED. It is not a viable candidate: the frame
# token sits inside the purpose sentence, so under causal attention its
# activation is a pure function of the frame text and cannot see the payload.
# Measured on the stored tensors, within-frame-group spread is 1.1% of overall
# spread and the 300 implicit items per language collapse to 20 distinct vectors.
# Its apparent advantage (0.90-0.95 balanced accuracy, contract-gate passes at
# layers 4-14) is a 20-point classification replicated 15x. See
# RESULT_controls_8b.md. The old tensors are kept locally so the artifact stays
# reproducible; there is no reason to spend a third of the archive re-storing it.
#
# The general rule this leaves behind: any candidate read-out position that sits
# BEFORE the payload has effective n = number of distinct frames, not number of
# items. Check effective n before trusting a new position.
POSITIONS = ["last", "mean"]

# The lexical controls test whether the direction rides the scrutiny-versus-use
# verbs rather than purpose attribution. At 8B they INVERT on both models: 27/40
# cells below chance at `last`, median ordering AUC 0.333, while explicit_lexeme
# passes 120/120. That is settled; they are re-extracted here only so each run is
# self-contained.
#
# KNOWN LIMITATION, do not over-read the controls: all 12 implicit_cue control
# terms (en: apply, check, employ, inspect, use, vet; ja: 利用 活用 点検 用い 確認
# 精査) are EXACTLY the differing cue spans of the fitting frames -- 12/12 overlap.
# So the controls establish that the direction follows those specific words. They
# cannot separate "encodes audience scrutiny abstractly" from "learned the frame
# verb vocabulary". They also cover NO agent nouns, so the noun_only pairs -- the
# ones the variants above rewrite -- currently have no manipulation check at all.
# See docs/CONTROL_COVERAGE_GAP.md.
EXTRACT_CONTROLS = True

# naturalistic_transfer.jsonl is deliberately NOT loaded. The frozen protocol
# allows that set to be scored once, after the probe and every analytical choice
# are final. Nothing in the current question needs it, and pulling it now only
# creates opportunities to spend it early.

BATCH_SIZE   = 8          # drop to 2 for a 70B in 4-bit
MAX_LENGTH   = 1024
LOAD_IN_4BIT = False      # True for 70B, or 8B on a T4
DTYPE        = "bfloat16" # ignored when LOAD_IN_4BIT

_est = 1200 * len(ITEM_SETS) * 4096 * 2 * len(LAYERS) * len(POSITIONS) / 1e6
print("models:", *MODELS, sep="\n  ")
print("item sets:", *[f"{n}  -> out/<model>{tag}" for tag, n in ITEM_SETS], sep="\n  ")
print(f"layers    : {LAYERS}")
print(f"positions : {POSITIONS}   ('frame' dropped: demonstrated artifact)")
print(f"controls  : {EXTRACT_CONTROLS}   (naturalistic transfer stays sealed)")
print(f"~{_est:.0f} MB per model before compression, "
      f"~{_est*len(MODELS)/1000:.1f} GB total")

In [ ]:
#@title 3. Load inputs

# Drop the files into Colab's file browser (folder icon, left sidebar) at the
# root -- drag and drop, or its upload button. The files.upload() widget below
# is only a fallback: it breaks whenever the cell output was restored from a
# saved notebook rather than produced live, and when third-party cookies are
# blocked. Drag-and-drop has neither failure mode.

import os, json, yaml
from pathlib import Path
from collections import Counter

REQUIRED = ["purpose_frames.yaml", ITEM_SETS[0][1]]
OPTIONAL = [name for _, name in ITEM_SETS[1:]]
if EXTRACT_CONTROLS:
    REQUIRED.append("lexical_controls.jsonl")
# Drive has to be mounted BEFORE it is searched. On a fresh runtime nothing has
# mounted it yet -- cell 5 does, but that runs after extraction -- so without
# this the Drive entry in SEARCH can never match and the documented "put the
# files in Drive once" workflow silently falls through to the upload widget.
# Best-effort: off Colab this is a no-op and the local paths still work.
def ensure_drive():
    if Path("/content/drive/MyDrive").is_dir():
        return "already mounted"
    try:
        from google.colab import drive
    except ImportError:
        return "not running on Colab -- using local paths only"
    try:
        drive.mount("/content/drive")
        return "mounted"
    except Exception as e:
        return f"mount failed ({type(e).__name__}: {e}) -- using local paths only"

print("Drive:", ensure_drive())

# Searched in order; first hit wins. Add a Drive path here if you sync the repo.
SEARCH = [Path("."), Path("/content"), Path("/content/drive/MyDrive/lang-ea")]

def locate(name):
    for d in SEARCH:
        p = d / name
        if p.is_file():
            return p
    return None

found = {n: locate(n) for n in REQUIRED + OPTIONAL}
missing = [n for n in REQUIRED if found[n] is None]

if missing:
    print("Missing:", ", ".join(missing))
    print("Drag them into the sidebar file browser, then re-run this cell.")
    try:
        from google.colab import files
        files.upload()                       # fallback; may be unavailable
        found = {n: locate(n) for n in REQUIRED + OPTIONAL}
        missing = [n for n in REQUIRED if found[n] is None]
    except Exception as e:
        print(f"upload widget unavailable ({type(e).__name__}) -- use the sidebar")
    if missing:
        raise FileNotFoundError(f"still missing: {missing}")

for n in REQUIRED + OPTIONAL:
    p = found[n]
    if p is None:
        print(f"  {n}: NOT FOUND -- this item set will be skipped")
    else:
        print(f"  {n}: {p.resolve()}  ({p.stat().st_size/1e3:.0f} kB)")

frames = {r["pair_id"]: r
          for r in yaml.safe_load(open(found["purpose_frames.yaml"], encoding="utf-8"))["pairs"]}

def load_items(path):
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    rows = [r for r in rows if r["source"] == "matched_generated"]
    rows.sort(key=lambda r: r["item_id"])
    return rows

item_sets = []
for tag, name in ITEM_SETS:
    if found.get(name) is None:
        continue
    rows = load_items(found[name])
    item_sets.append((tag, name, rows))
    print(f"\n{name}  ({len(rows)} items)  -> out/<model>{tag}")
    print("  by language :", dict(Counter(r["language"] for r in rows)))
    print("  by class    :", dict(Counter(r["class"] for r in rows)))
    print("  by locus    :", dict(Counter(frames[r["pair_id"]]["cue_locus"] for r in rows)))

# The item sets must differ ONLY in the frame line, or the before/after is not a
# controlled comparison. Check it here rather than discovering it afterwards.
if len(item_sets) > 1:
    base = {r["item_id"]: r["text"] for r in item_sets[0][2]}
    for tag, name, rows in item_sets[1:]:
        changed = same_payload = 0
        for r in rows:
            old = base.get(r["item_id"])
            if old is None:
                raise ValueError(f"{name}: item {r['item_id']} not in the base set")
            if old != r["text"]:
                changed += 1
                if old.split("\n", 1)[1:] == r["text"].split("\n", 1)[1:]:
                    same_payload += 1
        print(f"\n{name}: {changed} items differ from the base set, "
              f"{same_payload} of them ONLY in the first line")
        if changed != same_payload:
            raise ValueError(f"{name}: {changed - same_payload} items differ "
                             f"beyond the frame line -- not a controlled comparison")

controls = []
if EXTRACT_CONTROLS:
    controls = [json.loads(l) for l in open(found["lexical_controls.jsonl"], encoding="utf-8")]
    controls.sort(key=lambda r: r["control_id"])
    print(f"\ncontrol items: {len(controls)}")
    print("  by cue family :", dict(Counter(r["cue_family"] for r in controls)))
    print("  by type       :", dict(Counter(r["control_type"] for r in controls)))
    # The contract expects 8 crossed explicit_lexeme and 6 crossed implicit_cue
    # per language; check it here so a truncated file is caught before the GPU
    # time is spent rather than after.
    for cf, want in (("explicit_lexeme", 8), ("implicit_cue", 6)):
        for lg in ("en", "ja"):
            got = sum(1 for r in controls if r["cue_family"] == cf
                      and r["language"] == lg and r["control_type"] == "crossed_distractor")
            print(f"    crossed {cf:16} {lg}: {got} (contract expects {want})"
                  f"  {'ok' if got == want else 'MISMATCH'}")

In [ ]:
#@title 4. Extract
import gc, time, json, torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

FIT_KEYS = ["item_id", "pair_id", "language", "class", "label", "explicitness",
            "frame_family_id", "surface", "job_id", "content_sha256",
            "content_spec_id", "topic_id"]
CTL_KEYS = ["control_id", "control_pair_id", "control_type", "cue_family",
            "cue_term", "intended_purpose", "lexical_cue", "language"]


def frame_indices(tok_out, texts, items, L):
    """Token index of the end of the purpose-frame sentence, per row.

    Uses the fast tokenizer's character offsets rather than re-tokenizing a
    prefix. Re-tokenization is unreliable at exactly this boundary: the frame's
    final '.' / '。' merges with the following newlines and the first payload
    tokens into a single token, so no token ends where the sentence ends. The
    read-out is therefore the last token lying WHOLLY inside the frame sentence
    -- its final content token -- which is what we want anyway, since the
    straddling punctuation token already carries payload text.

    Returns (indices, n_fallback) where a fallback row is one whose frame could
    not be located and which falls back to the last token.
    """
    offsets = tok_out["offset_mapping"]
    idx, fallback = [], 0
    for bi, (r, text) in enumerate(zip(items, texts)):
        line = r["text"].split("\n")[0]
        at = text.find(line) if line.strip() else -1
        if at < 0:
            idx.append(L - 1)
            fallback += 1
            continue
        char_end = at + len(line)
        ends = offsets[bi, :, 1]
        # last token that both ends at/before the frame end and is a real token
        ok = ((ends <= char_end) & (ends > 0)).nonzero()
        if len(ok) == 0:
            idx.append(L - 1)
            fallback += 1
        else:
            idx.append(int(ok[-1]))
    return idx, fallback


def run_set(model, tok, layer_ids, items, out, meta_keys, meta_extra=None,
            use_frame=True, tag=""):
    """One forward pass over `items`, storing every (layer, position) slot."""
    out.mkdir(parents=True, exist_ok=True)
    feats = {(l, p): [] for l in layer_ids for p in POSITIONS}
    meta_f = open(out / "meta.jsonl", "w", encoding="utf-8")
    want_frame = "frame" in POSITIONS and use_frame
    t0, total_fallback = time.time(), 0

    with torch.inference_mode():
        for s in range(0, len(items), BATCH_SIZE):
            chunk = items[s : s + BATCH_SIZE]
            texts = [tok.apply_chat_template([{"role": "user", "content": r["text"]}],
                                             tokenize=False, add_generation_prompt=True)
                     for r in chunk]
            enc = tok(texts, return_tensors="pt", padding=True, truncation=True,
                      max_length=MAX_LENGTH,
                      return_offsets_mapping=want_frame)
            offs = enc.pop("offset_mapping", None)   # never goes to the model
            L = enc["attention_mask"].shape[1]
            if want_frame:
                fi, nfb = frame_indices({"offset_mapping": offs}, texts, chunk, L)
                total_fallback += nfb
            else:
                fi = [L - 1] * len(chunk)
            fidx = torch.tensor(fi, device=model.device)
            arange = torch.arange(len(chunk), device=model.device)

            enc = {k: v.to(model.device) for k, v in enc.items()}
            hs = model(**enc, output_hidden_states=True).hidden_states
            mask = enc["attention_mask"]

            for l in layer_ids:
                h = hs[l]
                m = mask.unsqueeze(-1).to(h.dtype)
                for p in POSITIONS:
                    if p == "last":
                        v = h[:, -1]                      # left padding: true last token
                    elif p == "mean":
                        v = (h * m).sum(1) / m.sum(1)     # non-padding tokens only
                    else:
                        v = h[arange, fidx]
                    feats[(l, p)].append(v.float().cpu())

            for r in chunk:
                rec = {k: r[k] for k in meta_keys if k in r}
                if meta_extra:
                    rec.update(meta_extra(r))
                meta_f.write(json.dumps(rec, ensure_ascii=False) + "\n")

            done = min(s + BATCH_SIZE, len(items))
            if done % 200 < BATCH_SIZE or done == len(items):
                el = time.time() - t0
                print(f"    {tag}{done}/{len(items)}  {el:.0f}s"
                      f"  eta {el/done*(len(items)-done):.0f}s", flush=True)
    meta_f.close()

    if want_frame and total_fallback:
        print(f"    WARNING: {total_fallback}/{len(items)} rows could not locate their "
              f"frame sentence and fell back to the last token", flush=True)

    for (l, p), parts in feats.items():
        torch.save(torch.cat(parts).half(), out / f"acts_layer{l}_{p}.pt")
    return len(items), total_fallback


def extract(model_id):
    slug = model_id.split("/")[-1]

    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = "left"          # keeps the final column the true last token
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    if "frame" in POSITIONS and not tok.is_fast:
        raise RuntimeError(f"{slug}: the 'frame' position needs a fast tokenizer "
                           f"for character offsets; drop it from POSITIONS")

    kw = dict(device_map="auto", low_cpu_mem_usage=True)
    if LOAD_IN_4BIT:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    else:
        kw["dtype"] = getattr(torch, DTYPE)
    model = AutoModelForCausalLM.from_pretrained(model_id, **kw).eval()

    n_layers = model.config.num_hidden_layers
    layer_ids = sorted({l for l in LAYERS if 1 <= l <= n_layers})
    dropped = sorted(set(LAYERS) - set(layer_ids))
    print(f"{slug}: {n_layers} layers, extracting {layer_ids}"
          + (f"  (dropped, out of range: {dropped})" if dropped else ""), flush=True)
    t0 = time.time()

    # One model load, every item set. Controls attach to the first set only --
    # their text is frame-independent, so extracting them per set would just be
    # identical activations stored under a second name.
    for set_tag, set_name, set_rows in item_sets:
        out = Path("out") / (slug + set_tag)
        n_fit, n_fb = run_set(model, tok, layer_ids, set_rows, out, FIT_KEYS,
                              meta_extra=lambda r: {"cue_locus": frames[r["pair_id"]]["cue_locus"]},
                              use_frame=True, tag=f"{set_name} ")
        n_ctl = 0
        if EXTRACT_CONTROLS and controls and set_tag == item_sets[0][0]:
            # Controls carry no purpose-frame sentence, so `frame` falls back to
            # `last` for them. Recorded in config.json so the scorer never
            # silently compares two different read-out positions.
            n_ctl, _ = run_set(model, tok, layer_ids, controls, out / "controls", CTL_KEYS,
                               use_frame=False, tag="controls ")

        json.dump({"model": model_id, "n_layers": n_layers, "layers_stored": layer_ids,
                   "positions": POSITIONS, "item_set": set_name,
                   "n_items": n_fit, "n_control_items": n_ctl,
                   "frame_position_fallbacks": n_fb,
                   "controls_frame_position_is_last": True,
                   "dtype": "float16", "max_length": MAX_LENGTH,
                   "load_in_4bit": LOAD_IN_4BIT},
                  open(out / "config.json", "w"), indent=2)
        print(f"  saved -> {out}  ({n_fit} items + {n_ctl} control, "
              f"{n_fb} frame fallbacks)", flush=True)
    print(f"  {slug} done in {time.time()-t0:.0f}s")

    del model; gc.collect(); torch.cuda.empty_cache()


for m in MODELS:
    try:
        extract(m)
    except Exception as e:
        print(f"FAILED {m}: {type(e).__name__}: {e}\n")

In [ ]:
#@title 5. Zip and save to Drive

# files.download() cannot reach the local filesystem from the VS Code Colab
# extension (googlecolab/colab-vscode#276), so Drive is the transport. Same
# folder cell 3 searches for inputs, so a re-run finds everything in one place.

import os, shutil
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/lang-ea")

if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

shutil.make_archive("activations", "zip", "out")
size_mb = os.path.getsize("activations.zip") / 1e6
print(f"activations.zip  {size_mb:.1f} MB")

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
dest = DRIVE_DIR / "activations.zip"
if dest.exists():
    print(f"overwriting existing {dest} ({dest.stat().st_size/1e6:.1f} MB)")
shutil.copy("activations.zip", dest)
print(f"saved -> {dest}")

# Browser Colab only; harmless no-op under the VS Code extension.
try:
    from google.colab import files
    files.download("activations.zip")
except Exception as e:
    print(f"direct download unavailable ({type(e).__name__}) -- pull it from Drive")

---
### Back on the local machine

Unzip into `contrastive/probe_check/out/`, then per model directory:

```powershell
# the load-bearing one: does the direction follow purpose, or the cue word?
uv run --no-project --with torch --with scikit-learn --with numpy `
  python probe_check/analyze.py --what controls --layer 8 `
  --acts probe_check/out/Llama-3.1-Swallow-8B-Instruct-v0.3

# layer and token-position sweep, with the contract's inner selection
uv run --no-project --with torch --with scikit-learn --with numpy `
  python probe_check/analyze.py --what folds --inner-cv --position last `
  --acts probe_check/out/Llama-3.1-Swallow-8B-Instruct-v0.3
```

Then the rewrite, which needs both item sets:

```powershell
# did catalog 2.3.2 fix noun_only? compare the two directories at the same layer
uv run --no-project --with torch --with scikit-learn --with numpy `
  python probe_check/analyze.py --what folds --position last `
  --acts probe_check/out/Meta-Llama-3.1-8B-Instruct
uv run --no-project --with torch --with scikit-learn --with numpy `
  python probe_check/analyze.py --what folds --position last `
  --acts probe_check/out/Meta-Llama-3.1-8B-Instruct_newframes
```

Read the `noun_only` row of the locus table in each, and the effect size under `--what nulls`. Effect size is the more reliable comparison: the fitted direction shifts slightly between the two runs because the training set changed, which moves even the byte-identical `both` and `verb_only` items by up to 0.06 balanced accuracy. Treat that as the noise floor.

`--position` takes `last`, `mean` or `frame`; `--what` takes `folds`, `locus`, `paired`, `nulls` or `controls`; `--inner-cv` adds the contract's group-aware inner selection. `fit.py` still runs the original protocol unchanged.

**What this run is for.** The first run answered the question that was blocking the cue-locus decision and raised two new ones, both of which this run is built to settle:

1. **Is the direction reading purpose, or the scrutiny-versus-use verbs?** Every cue locus that decoded well carried a verb contrast, and the only locus isolating the agent noun failed. That is the signature of a verb-driven direction. The `implicit_cue` lexical controls are the instrument that separates the two, and they had never been extracted. `--what controls` scores them against the contract's own numbers: crossed distractors must follow intended purpose rather than the cue word (≥7/8 explicit, ≥5/6 implicit, per language), and mention-only items must move the projection by less than 0.25 SD.

2. **Which layer and token position get frozen?** Layer 8 of 32 won every metric last time and was the only depth to pass the per-fold gate — but it was the shallowest of four points sampled, so the optimum may be outside the sampled set entirely. This run samples the shallow end densely and stores three read-out positions instead of one.

**What it is still not.** `demo16_v2` is a development corpus the frozen protocol forbids reusing, and these are not the declared subjects. Nothing produced here is gate evidence. `naturalistic_transfer.jsonl` is deliberately never loaded: the protocol allows it to be scored once, after every analytical choice is frozen.